# Handling Imbalanced Data

**Topic:** Data Preprocessing

In [ ]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import Dropdown, Output, VBox
from IPython.display import display, clear_output

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils import resample

import seaborn as sns
titanic = sns.load_dataset("titanic")

from tkh_utils import PALETTE, FONT, base_layout

np.random.seed(42)

# Minimal preprocessing so we can run a classifier for the demo.
# We are not teaching these steps here; they were covered in notebooks 02–04.
df = titanic[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()
df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["fare"]     = df["fare"].fillna(df["fare"].median())
df = pd.get_dummies(df, columns=["sex", "embarked"], drop_first=False)

X = df.drop(columns="survived").values.astype(float)
y = df["survived"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

---
## What you'll explore

By the end of this notebook you will be able to:

- **Describe** what class imbalance is and why accuracy alone is a misleading metric when classes are unequal in size
- **Explain** the trade-offs between undersampling, oversampling, SMOTE, and class weighting
- **Interpret** precision, recall, and F1 score in the context of an imbalanced classification problem

> **Tip:** Select "None" first and look at the accuracy. It looks reasonable, but look at recall. Now try SMOTE and compare all four metrics together.

---
## How we got here

In `06_train_test_split.ipynb` we learned to hold out a test set and why stratification matters. Now we look at what happens when the classes in the dataset are not evenly distributed.

The Titanic's survival rate is 38%: only 342 of 891 passengers survived. The classes are imbalanced: there are 1.6 non-survivors for every survivor. This is not the most extreme imbalance you will encounter (fraud detection is typically 99% vs 1%), but it is enough to demonstrate the core problem.

---
## Why this matters for data science

In medical diagnosis, fraud detection, and rare event prediction, the minority class is usually the one you care about most. A model that predicts "no fraud" for every transaction can achieve 99% accuracy on a dataset where 1% of transactions are fraudulent, and it learns nothing useful. Imbalanced data turns accuracy into a trap.

---
## The accuracy trap

In [ ]:
survived_count    = int(y.sum())
not_survived      = int((y == 0).sum())
survival_rate     = y.mean()

print(f"Total passengers:    {len(y)}")
print(f"Survived (class 1):  {survived_count}  ({survival_rate:.1%})")
print(f"Did not survive (0): {not_survived}  ({1 - survival_rate:.1%})")
print()
print("A model that always predicts 'did not survive' would score:")
print(f"  Accuracy: {1 - survival_rate:.1%}  ← looks decent, but tells us nothing useful")

---
## Try it yourself

In [ ]:
out_bar = Output()

def balance_data(X_train, y_train, technique):
    if technique == "None (original)":
        return X_train.copy(), y_train.copy()

    X_df = pd.DataFrame(X_train)
    X_df["label"] = y_train

    majority = X_df[X_df["label"] == 0]
    minority = X_df[X_df["label"] == 1]

    if technique == "Undersample majority":
        maj_down = resample(majority, replace=False,
                            n_samples=len(minority), random_state=42)
        result = pd.concat([maj_down, minority])

    elif technique == "Oversample minority":
        min_up = resample(minority, replace=True,
                          n_samples=len(majority), random_state=42)
        result = pd.concat([majority, min_up])

    elif technique == "SMOTE (synthetic)":
        try:
            from imblearn.over_sampling import SMOTE
            sm = SMOTE(random_state=42)
            X_out, y_out = sm.fit_resample(X_train, y_train)
            return X_out, y_out
        except ImportError:
            # Fallback to simple oversampling if imbalanced-learn is not installed
            min_up = resample(minority, replace=True,
                              n_samples=len(majority), random_state=42)
            result = pd.concat([majority, min_up])

    X_out = result.drop(columns="label").values
    y_out = result["label"].values
    return X_out, y_out

def get_metrics(X_train, y_train, X_test, y_test):
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return {
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall":    recall_score(y_test, y_pred, zero_division=0),
        "F1 Score":  f1_score(y_test, y_pred, zero_division=0),
    }

TECHNIQUES = ["None (original)", "Undersample majority",
              "Oversample minority", "SMOTE (synthetic)"]

tech_dd = Dropdown(
    options=TECHNIQUES,
    value="None (original)",
    description="Technique:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="320px"),
)

def render(change=None):
    technique = tech_dd.value
    X_bal, y_bal = balance_data(X_tr, y_tr, technique)
    metrics = get_metrics(X_bal, y_bal, X_te, y_te)

    train_counts = [int((y_bal == 0).sum()), int((y_bal == 1).sum())]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Class Distribution (training set)", "Model Performance on Test Set"],
    )

    fig.add_trace(go.Bar(
        x=["Did not survive", "Survived"],
        y=train_counts,
        marker_color=[PALETTE["primary"], PALETTE["secondary"]],
        showlegend=False,
        text=[str(c) for c in train_counts],
        textposition="outside",
    ), row=1, col=1)

    metric_names  = list(metrics.keys())
    metric_values = list(metrics.values())
    bar_colors = [
        PALETTE["accent"] if v >= 0.75
        else PALETTE["secondary"] if v < 0.60
        else PALETTE["muted"]
        for v in metric_values
    ]

    fig.add_trace(go.Bar(
        x=metric_names, y=metric_values,
        marker_color=bar_colors,
        showlegend=False,
        text=[f"{v:.3f}" for v in metric_values],
        textposition="outside",
    ), row=1, col=2)

    fig.update_layout(
        title=dict(
            text=f"Balancing Technique: {technique}",
            font=dict(size=FONT["size_title"], family=FONT["family"]),
        ),
        paper_bgcolor=PALETTE["background"],
        plot_bgcolor=PALETTE["surface"],
        height=420,
        margin=dict(l=50, r=20, t=80, b=50),
    )
    fig.update_yaxes(title_text="Count", row=1, col=1)
    fig.update_yaxes(title_text="Score", range=[0, 1.1], row=1, col=2)

    with out_bar:
        clear_output(wait=True)
        display(go.FigureWidget(data=fig.data, layout=fig.layout))

tech_dd.observe(render, names="value")
display(VBox([tech_dd, out_bar]))
render()

---
## What's happening?

Class imbalance occurs when one label appears significantly more often than another. When a model trains on imbalanced data without any correction, it learns that predicting the majority class is almost always correct, so it stops learning the minority class signal.

**Undersampling** removes rows from the majority class until both classes are equal. It is fast and simple, but throws away real training data.

**Oversampling** duplicates rows from the minority class. No data is lost, but the model sees exact duplicates, which can cause overfitting.

**SMOTE** (Synthetic Minority Oversampling Technique) creates new synthetic minority examples by interpolating between existing ones. It avoids duplicates while still expanding the minority class. It requires the `imbalanced-learn` library.

**Class weighting** does not change the data at all: it tells the algorithm to penalize errors on the minority class more heavily. Some algorithms support a `class_weight` parameter directly.

> **Discussion question:** In the Titanic context, which error is worse: predicting a survivor died, or predicting someone died when they actually survived? How does that affect which metric you optimize for?

| Technique | How it works | Risk | When to use it |
|---|---|---|---|
| Undersampling | Remove majority class rows | Loses training data | When dataset is large enough to afford it |
| Oversampling | Duplicate minority class rows | Can cause overfitting | Small datasets with mild imbalance |
| SMOTE | Synthesize new minority examples | May create unrealistic samples | Moderate imbalance, tabular data |
| Class weights | Penalize minority class errors more | Less direct than resampling | When resampling is not an option |

### Model-dependent note

Some algorithms accept a `class_weight='balanced'` parameter directly. Others require resampling at the data level before training. You will see both patterns when you meet your first algorithms. The chart below shows how adjusting class weights changes the trade-off between precision and recall.

---
## Real-world example: A cancer screening model that predicted "no cancer" every time

A radiology team built a model to flag suspicious mammograms. Their dataset was 93% negative and 7% positive. The model achieved 93% accuracy by predicting "no cancer" for every patient. Recall for the positive class was 0.00. After applying SMOTE and switching from accuracy to F1 score as the primary metric, the model reached 78% recall on true positives, catching most of the cases that mattered.

In [ ]:
np.random.seed(42)
weights = [None, {0: 1, 1: 2}, {0: 1, 1: 5}, {0: 1, 1: 10}]
labels  = ["No weighting", "2x minority weight", "5x minority weight", "10x minority weight"]
results = []

for w in weights:
    clf = LogisticRegression(max_iter=1000, random_state=42, class_weight=w)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    results.append({
        "Accuracy":  accuracy_score(y_te, y_pred),
        "Recall":    recall_score(y_te, y_pred, zero_division=0),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "F1":        f1_score(y_te, y_pred, zero_division=0),
    })

fig = go.Figure()
for metric, color in [
    ("Accuracy",  PALETTE["muted"]),
    ("Recall",    PALETTE["secondary"]),
    ("Precision", PALETTE["primary"]),
    ("F1",        PALETTE["accent"]),
]:
    fig.add_trace(go.Bar(
        name=metric,
        x=labels,
        y=[r[metric] for r in results],
        marker_color=color,
    ))

layout = base_layout(
    title="Effect of Class Weights on Metrics: Accuracy Hides What Recall Reveals",
    xaxis_title="Class weight setting",
    yaxis_title="Score",
)
layout.update(barmode="group", yaxis=dict(range=[0, 1.1]))
go.FigureWidget(data=fig.data, layout=layout)

---
## Key takeaway

> **High accuracy on an imbalanced dataset usually means the model learned to predict the majority class. Always check precision, recall, and F1 before declaring a model successful.**

---
*Next up: 08_data_leakage.ipynb. It is the most common reason a model that looks great in development fails completely in production*